In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchtune.modules import RotaryPositionalEmbeddings
from torch.optim.lr_scheduler import OneCycleLR
from torch.amp import autocast, GradScaler
from datasets import load_dataset, load_from_disk
import json
import re
from collections import Counter
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
running on cuda


In [5]:
vocab_size = 0
mask_id = 0
pad_id = 1
unknown_id = 2
min_freq = 0

def collate_fn(batch, pad_id):
    padded = torch.stack(batch)
    padding_mask = (padded == pad_id)
    return padded, padding_mask

collate_id = lambda batch: collate_fn(batch, pad_id)

class CaptionDataset(Dataset):
    def __init__(self, captions, vocab: dict, max_len=32, unknown_id = unknown_id):
        self.captions = captions
        self.vocab    = vocab
        self.max_len  = max_len

        tokens = []
        for cap in captions:
            words = cap.split()[:max_len]
            ids   = [vocab.get(w, unknown_id) for w in words]
            ids  += [pad_id] * (max_len - len(ids))
            tokens.append(ids)

        self.tokens = torch.tensor(tokens, dtype=torch.long) 

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        return self.tokens[idx]


try:
    ds = load_from_disk("datasets/facad_dataset")
except:
    ds = load_dataset("Luna288/image-captioning-FACAD-base", split="train")
    ds.save_to_disk("datasets/facad_dataset")
captions = [ex["text"] for ex in ds]

new_captions = []
counts = Counter()
for cap in captions:
    cap = cap.lower()
    cap = re.sub(r"[-/]", " ", cap)
    cap = re.sub(r"[^a-z0-9 ]", "", cap)
    new_captions.append(cap)
    counts.update(cap.split())

captions = new_captions

vocab = {"<mask>": 0, "<pad>": 1, "<unk>": 2}
for word, count in counts.items():
    if count >= min_freq:
        vocab[word] = len(vocab)
    
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

dataset = CaptionDataset(captions, vocab)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_id,
    num_workers=0,
    pin_memory=False,
)

print(captions[:10])

Saving the dataset (0/2 shards):   0%|          | 0/123653 [00:00<?, ? examples/s]

Vocab size: 17224
['flower fall and spot spatter this wear everywhere top cut with a gracefully pleated neckline', 'inspired by classic american style this prepster staple is knit from soft warming wool and sharpened up with colorful stripe and a crisp cotton twill collar', 'pretty floral lace bloom across this cropped camisole that s soft stretchy and comfortable to wear', 'if the sweater is clean so is the crisp white tee to top beneath it thanks to the sweater over tee look of this pullover style that s already paired', 'frilly trim enhances the whimsical appeal of this polka dot blouse fashioned from light luminous silk', 'rich metallic stripe elevate a handsome tie cut from pure silk', 'a high waist cut and a just right stretch combine to create a pair of everyday jeans that fit to flatter with comfort to spare', 'strap in and take the plunge with this satin dress that will make you feel like the life of the party', 'elegantly intricate lace add art deco vibe to a slender sheath d

In [7]:
class MultiHeadAttentionNode(nn.Module):
    def __init__(self, d_model, num_heads, dropout = 0.2):
        assert d_model % num_heads == 0
        super().__init__()

        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        self.num_heads = num_heads
        self.d_head = int(d_model / num_heads)
        self.scale = self.d_head ** 0.5

        self.rope = RotaryPositionalEmbeddings(int(d_model//num_heads))

    def forward(self, x, padding_mask = None):
        B, N, D = x.shape
        QKV = self.Wqkv(x)
        QKV = QKV.view(B, N, 3, self.num_heads, self.d_head)
        QKV = QKV.permute(2, 0, 3, 1, 4)  # (3, B, H, N, d_head)
        Q, K, V = QKV[0], QKV[1], QKV[2]

        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        Q = self.rope(Q).permute(0, 2, 1, 3)
        K = self.rope(K).permute(0, 2, 1, 3)

        score = (Q @ K.transpose(-2, -1)) # batch H N N
        score = score / self.scale

        if padding_mask is not None:
            # padding_mask is (B, N), need (B, 1, 1, N) to broadcast over heads
            score = score.masked_fill(padding_mask[:, None, None, :], float('-inf'))

        weights = self.dropout(F.softmax(score, dim=-1))
        out = torch.matmul(weights, V) # batch H N d_head

        out = out.transpose(1, 2).contiguous() # B N H d_head
        out = out.view(B, N, D)

        out = self.dropout(self.Wo(out))

        return out, weights

class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff, heads, dropout = 0.2):
        super().__init__()
        self.attn = MultiHeadAttentionNode(d_model, heads, dropout=dropout)
        self.ff = nn.Sequential(*[
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            
        ])
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, padding_mask=None):
        x = x + self.dropout(self.attn(self.ln1(x), padding_mask)[0])
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()

        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformers = nn.ModuleList(
            [TransformerBlock(d_model, d_ff, num_heads) for i in range(num_layers)]
        )
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x, padding_mask = None):
        out = self.embed(x)
        for transformer in self.transformers:
            out = transformer(out, padding_mask)
        norm = self.ln(out)
        return norm
    
class MLMTrainer(nn.Module):
    def __init__(self, encoder: Encoder, encoder_dim, vocab_size, mask_id, dropout = 0.2, mask_ratio = 0.15):
        super().__init__()

        self.encoder = encoder
        self.vocab_size = vocab_size
        self.decoder = nn.Linear(encoder_dim, vocab_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        self.mask_ratio = mask_ratio
        self.mask_id = mask_id

        self.decoder.weight = self.encoder.embed.weight

    def mask_tokens(self, x):
        mask = (torch.rand(x.shape, device=x.device) < self.mask_ratio) & (x != pad_id)
        decision_mask = torch.rand(x.shape, device=x.device)
        mask_mask = mask & (decision_mask < 0.8)
        unknown_mask = mask & (decision_mask > 0.8) & (decision_mask < 0.9)
        
        ground = x.clone()
        masked = x.clone()

        masked[mask_mask] = self.mask_id
        masked[unknown_mask] = torch.randint(3, self.vocab_size, masked[unknown_mask].shape, device = device)
        ground[~mask] = -100

        return masked, ground

    def forward(self, x, padding_mask = None):
        masked, ground = self.mask_tokens(x)
        pred = self.encoder(masked, padding_mask)
        return self.decoder(pred), ground

In [8]:
encoding_dim = 512

whereswaldo = Encoder(
    vocab_size=vocab_size,
    d_model=encoding_dim,
    num_heads=8,
    d_ff=4*encoding_dim,
    num_layers=8
).to(device)

whereswaldotrainer = MLMTrainer(
    encoder=whereswaldo,
    encoder_dim=encoding_dim,
    vocab_size=vocab_size,
    mask_id=mask_id
).to(device)

nn.init.normal_(whereswaldo.embed.weight, std=0.02)
optimizer = torch.optim.AdamW(whereswaldotrainer.parameters(), lr = 3e-4, weight_decay=0.1)
loss_fn = F.cross_entropy

ckpt = torch.load("trained/reconstructed_model.pt", map_location=device)
whereswaldotrainer.load_state_dict(ckpt["model"])

<All keys matched successfully>

In [ ]:
class SentenceEncoder(nn.Module):
    def __init__(self, encoder : Encoder, train_enc = False):
        super().__init__()
        self.encoder = encoder
        self.sentence_query = nn.Parameter(torch.randn(1, 1, encoder.d_model))
        self.KV = nn.Linear(encoder.d_model, 2*encoder.d_model)
        self.scale = encoder.d_model ** 0.5
        self.ff = nn.Linear(encoder.d_model, encoder.d_model)

        for param in self.encoder.parameters():
            param.requires_grad = train_enc

    def forward(self, x, padding_mask = None):
        ctx_tokens = self.encoder(x, padding_mask)
        b, n, d = ctx_tokens.shape
        KV = self.KV(ctx_tokens)
        K, V = KV.view(b, n, 2, d).contiguous().permute(2, 0, 1, 3)

        q = self.sentence_query.expand(b, -1, -1)
        scores = torch.matmul(q, K.transpose(-2, -1)) / self.scale
        
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask[:, None, :], float('-inf'))
        scores = F.softmax(scores, dim=-1)

        out = torch.matmul(scores, V)
        return F.normalize(self.ff(out.squeeze(1)))

class SEtrainer(nn.Module):
    def __init__(self, se : SentenceEncoder, mask_ratio = 0.5, mask_id = mask_id):
        super().__init__()        
        self.se = se
        self.mask_ratio = mask_ratio
        self.mask_id = mask_id

        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def mask_tokens(self, x):
        mask = (torch.rand(x.shape, device=x.device) < self.mask_ratio) & (x != pad_id)

        ground = x.clone()
        masked = x.clone()

        masked[mask] = self.mask_id
        ground[~mask] = -100

        return masked, ground

    def loss(self, x):
        B2, d = x.shape
        sims = x @ x.transpose(-2, -1)
        sims = sims * self.logit_scale.exp()
        mask = torch.eye(B2, device=x.device).bool()
        sims = sims.masked_fill(mask, -1e9)

        targets = torch.arange(int(B2//2), device=x.device)
        targets = torch.cat([targets + int(B2//2), targets])

        return F.cross_entropy(sims, targets)
    
    def forward(self, x, padding_mask = None):
        inp = torch.cat([x, x])
        inp, _ = self.mask_tokens(inp)
        embedded = self.se(inp, padding_mask)

        return embedded

